# Repository note
This is a code-only copy of an exploratory notebook. Saved outputs were removed for public sharing. The required CSV datasets are not included; update the local/Colab paths in the cells before running. The original execution environment and full dataset provenance were not supplied, so a fresh run is not guaranteed to reproduce prior results.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# **TCP**

In [ ]:
import pandas as pd

In [ ]:
# ── Load your TCP normal dataset ──────────────────────────────────────────────
tcp_normal = pd.read_csv("/content/normal_tcp.csv")

print(f"Original shape: {tcp_normal.shape}")

In [ ]:
print(f"Columns: {tcp_normal.columns.tolist()}")

# ── Step 1: Drop non-feature columns (IP, ID, timestamp etc.) ────────────────
non_feature_cols = [
    'flow_id', 'src_ip', 'dst_ip', 'source_ip', 'dest_ip',
    'timestamp', 'src_port', 'dst_port'
]
drop_cols = [col for col in non_feature_cols if col in tcp_normal.columns]
tcp_normal.drop(columns=drop_cols, inplace=True)
print(f"Dropped non-feature columns: {drop_cols}")

# ── Step 2: Drop any existing label columns ───────────────────────────────────
label_cols = [col for col in tcp_normal.columns if 'label' in col.lower()]
tcp_normal.drop(columns=label_cols, inplace=True)
print(f"Dropped label columns: {label_cols}")

print(f"Shape after dropping: {tcp_normal.shape}")

# ── Step 3: Balance to first 7,000 rows ──────────────────────────────────────
tcp_normal = tcp_normal.head(7000).reset_index(drop=True)
print(f"After balancing: {len(tcp_normal)} rows")

# ── Step 4: Verify we have 84 features ───────────────────────────────────────
print(f"\nTotal features available: {tcp_normal.shape[1]}")

# ── Step 5: Add 3 label columns ───────────────────────────────────────────────
tcp_normal['label_binary']    = 0      # Normal traffic
tcp_normal['label_protocol']  = 'TCP'  # Protocol
tcp_normal['label_technique'] = 0      # No technique (normal)

# ── Step 6: Verify ────────────────────────────────────────────────────────────
print(f"\n✅ Final shape: {tcp_normal.shape}")
print("\n✅ Label summary:")
print(tcp_normal[['label_binary', 'label_protocol', 'label_technique']].value_counts())
print("\n✅ Sample rows:")
print(tcp_normal[['label_binary', 'label_protocol', 'label_technique']].head(5))
print("\n✅ All columns:")
print(tcp_normal.columns.tolist())

# ── Step 7: Save ──────────────────────────────────────────────────────────────
tcp_normal.to_csv("tcp_normal_labeled.csv", index=False)
print("\n✅ Saved as tcp_normal_labeled.csv")

In [ ]:
import pandas as pd

# ── Load your TCP normal dataset ──────────────────────────────────────────────
tcp_normal = pd.read_csv("/content/normal_tcp.csv")

print(f"Original shape: {tcp_normal.shape}")

# ── Step 1: Drop any existing label columns (if present) ──────────────────────
label_cols = [col for col in tcp_normal.columns if 'label' in col.lower()]
tcp_normal.drop(columns=label_cols, inplace=True)
print(f"Dropped label columns: {label_cols}")

# ── Step 2: Balance to 7,000 rows (take first 7,000 rows) ────────────────────
tcp_normal = tcp_normal.head(7000).reset_index(drop=True)
print(f"After balancing: {tcp_normal.shape}")

# ── Step 3: Add the 3 label columns ──────────────────────────────────────────
tcp_normal["label_binary"]    = 0       # Normal traffic
tcp_normal["label_protocol"]  = "TCP"   # Protocol
tcp_normal["label_technique"] = 0       # No technique (normal)

# ── Step 4: Verify ────────────────────────────────────────────────────────────
print("\n✅ Final shape:", tcp_normal.shape)
print("✅ Label columns:\n", tcp_normal[["label_binary", "label_protocol", "label_technique"]].value_counts())
print("\n✅ Sample rows:")
print(tcp_normal[["label_binary", "label_protocol", "label_technique"]].head(5))

# ── Step 5: Save ──────────────────────────────────────────────────────────────
tcp_normal.to_csv("tcp_normal_labeled.csv", index=False)
print("\n✅ Saved as tcp_normal_labeled.csv")


In [ ]:
import numpy as np
from sklearn.preprocessing import StandardScaler

# ── Load TCP Covert Dataset ───────────────────────────────────────────────────
tcp_covert = pd.read_csv("/content/drive/MyDrive/Protocol-Wise/Protocol-Wise/TCP/tcp_covert.csv")
print(f"Original shape: {tcp_covert.shape}")

# ── Step 1: Drop any existing label columns ───────────────────────────────────
label_cols = [col for col in tcp_covert.columns if 'label' in col.lower()]
tcp_covert.drop(columns=label_cols, inplace=True)
print(f"Dropped label columns: {label_cols}")

# ── Step 2: Define Feature Groups ────────────────────────────────────────────
header_features = [
    'syn_count', 'ack_count', 'psh_count', 'fin_count', 'rst_count', 'urg_count',
    'syn_ratio', 'ack_ratio', 'psh_ratio', 'fin_ratio', 'rst_ratio', 'urg_ratio',
    'win_size_mean', 'win_size_std', 'win_size_min', 'win_size_max',
    'win_size_variance', 'win_size_entropy', 'win_size_changes', 'win_size_cv',
    'seq_mean_increment', 'seq_std_increment', 'seq_entropy', 'seq_large_jumps',
    'ack_unique_count', 'ack_entropy'
]

blending_features = [
    'pkt_len_mean', 'pkt_len_std', 'pkt_len_min', 'pkt_len_max',
    'pkt_len_median', 'pkt_len_variance', 'pkt_len_entropy', 'pkt_len_cv',
    'payload_len_mean', 'payload_len_std', 'payload_len_min',
    'payload_len_max', 'payload_len_entropy',
    'fwd_pkt_len_mean', 'fwd_pkt_len_std', 'fwd_pkt_len_min', 'fwd_pkt_len_max',
    'bwd_pkt_len_mean', 'bwd_pkt_len_std', 'bwd_pkt_len_min', 'bwd_pkt_len_max',
    'total_bytes', 'fwd_bytes', 'bwd_bytes',
    'fwd_bwd_pkt_ratio', 'fwd_bwd_bytes_ratio',
    'total_packets', 'fwd_packets', 'bwd_packets',
    'retransmission_count', 'retransmission_ratio',
    'segment_not_captured_count', 'segment_not_captured_ratio'
]

timing_features = [
    'duration', 'pkt_rate', 'fwd_pkt_rate', 'bwd_pkt_rate', 'burstiness',
    'iat_mean', 'iat_std', 'iat_min', 'iat_max', 'iat_median',
    'iat_variance', 'iat_cv', 'iat_entropy', 'iat_skewness', 'iat_kurtosis',
    'fwd_iat_mean', 'fwd_iat_std', 'fwd_iat_min', 'fwd_iat_max',
    'bwd_iat_mean', 'bwd_iat_std', 'bwd_iat_min', 'bwd_iat_max'
]

# ── Step 3: Filter to only available columns (safety check) ──────────────────
header_features   = [f for f in header_features   if f in tcp_covert.columns]
blending_features = [f for f in blending_features if f in tcp_covert.columns]
timing_features   = [f for f in timing_features   if f in tcp_covert.columns]

print(f"\nUsing features → Header: {len(header_features)} | "
      f"Blending: {len(blending_features)} | Timing: {len(timing_features)}")

# ── Step 4: Standardize features ─────────────────────────────────────────────
scaler = StandardScaler()
all_features = header_features + blending_features + timing_features
tcp_scaled = pd.DataFrame(
    scaler.fit_transform(tcp_covert[all_features]),
    columns=all_features
)

# ── Step 5: Compute Anomaly Score per Group (mean of absolute z-scores) ───────
tcp_covert['score_header']   = tcp_scaled[header_features].abs().mean(axis=1)
tcp_covert['score_blending'] = tcp_scaled[blending_features].abs().mean(axis=1)
tcp_covert['score_timing']   = tcp_scaled[timing_features].abs().mean(axis=1)

# ── Step 6: Assign technique based on highest anomaly score ──────────────────
# Technique 1 = Header Manipulation
# Technique 2 = Timing Obfuscation
# Technique 3 = Flow Blending

def assign_technique(row):
    scores = {
        1: row['score_header'],
        2: row['score_timing'],
        3: row['score_blending']
    }
    return max(scores, key=scores.get)

tcp_covert['label_technique'] = tcp_covert.apply(assign_technique, axis=1)

# ── Step 7: Add remaining label columns ──────────────────────────────────────
tcp_covert['label_binary']   = 1
tcp_covert['label_protocol'] = 'TCP'

# ── Step 8: Drop scoring helper columns ──────────────────────────────────────
tcp_covert.drop(columns=['score_header', 'score_blending', 'score_timing'], inplace=True)

# ── Step 9: Verify ────────────────────────────────────────────────────────────
print("\n✅ Final shape:", tcp_covert.shape)
print("\n✅ Technique distribution:")
print(tcp_covert['label_technique'].value_counts().sort_index())
print("\n✅ Label summary:")
print(tcp_covert[['label_binary', 'label_protocol', 'label_technique']].value_counts())
print("\n✅ Sample rows:")
print(tcp_covert[['label_binary', 'label_protocol', 'label_technique']].head(10))

# ── Step 10: Save ─────────────────────────────────────────────────────────────
tcp_covert.to_csv("tcp_covert_labeled.csv", index=False)
print("\n✅ Saved as tcp_covert_labeled.csv")

# **ICMP**


In [ ]:
icmp_normal = pd.read_csv("/content/drive/MyDrive/Protocol-Wise/Protocol-Wise/ICMP/ICMP_Normal_Traffic.csv")
icmp_normal.head()

In [ ]:
icmp_normal.columns.tolist()

In [ ]:

# ── Step 1: Drop non-feature columns ─────────────────────────────────────────
drop_cols = ['flow_id', 'source_ip', 'dest_ip', 'label']
drop_cols = [col for col in drop_cols if col in icmp_normal.columns]
icmp_normal.drop(columns=drop_cols, inplace=True)
print(f"Dropped columns: {drop_cols}")

# ── Step 2: Check remaining shape ─────────────────────────────────────────────
print(f"Shape after dropping: {icmp_normal.shape}")
print(f"Total rows: {len(icmp_normal)}")

# ── Step 3: Balance rows ──────────────────────────────────────────────────────
# ⚠️ Change 7000 to however many rows you want to balance to
BALANCE_TO = 7000

if len(icmp_normal) >= BALANCE_TO:
    icmp_normal = icmp_normal.head(BALANCE_TO).reset_index(drop=True)
    print(f"Balanced to: {len(icmp_normal)} rows")
else:
    print(f"⚠️ Only {len(icmp_normal)} rows available — using all (less than {BALANCE_TO})")
    icmp_normal = icmp_normal.reset_index(drop=True)

# ── Step 4: Add 3 label columns ───────────────────────────────────────────────
icmp_normal['label_binary']    = 0        # Normal traffic
icmp_normal['label_protocol']  = 'ICMP'   # Protocol
icmp_normal['label_technique'] = 0        # No technique (normal)

# ── Step 5: Verify ────────────────────────────────────────────────────────────
print(f"\n✅ Final shape: {icmp_normal.shape}")
print("\n✅ Label summary:")
print(icmp_normal[['label_binary', 'label_protocol', 'label_technique']].value_counts())
print("\n✅ Sample rows:")
print(icmp_normal[['label_binary', 'label_protocol', 'label_technique']].head(5))

# ── Step 6: Save ──────────────────────────────────────────────────────────────
icmp_normal.to_csv("icmp_normal_labeled.csv", index=False)
print("\n✅ Saved as icmp_normal_labeled.csv")

In [ ]:
#ICMP-COVERT


icmp_covert = pd.read_csv("/content/drive/MyDrive/Protocol-Wise/Protocol-Wise/ICMP/ICMP_Covert_Traffic.csv")
icmp_covert.head(3)

In [ ]:
print(f"Original shape: {icmp_covert.shape}")

# ── Step 1: Check existing label values ───────────────────────────────────────
print(f"\nExisting label distribution:")
print(icmp_covert['label'].value_counts())

# ── Step 2: Map existing label → label_technique (1, 2, 3) ───────────────────
# ICMP_Header_Manipulation → 1
# ICMP_Timing_Obfuscation  → 2
# ICMP_Flow_Bending        → 3

technique_map = {
    'ICMP_Header_Manipulation' : 1,
    'ICMP_Timing_Obfuscation'  : 2,
    'ICMP_Flow_Bending'        : 3
}

icmp_covert['label_technique'] = icmp_covert['label'].map(technique_map)

# ── Step 3: Check for any unmapped values ─────────────────────────────────────
unmapped = icmp_covert['label_technique'].isna().sum()
if unmapped > 0:
    print(f"\n⚠️ Warning: {unmapped} rows could not be mapped — check label values!")
    print(icmp_covert[icmp_covert['label_technique'].isna()]['label'].unique())
else:
    print(f"\n✅ All rows mapped successfully!")

# ── Step 4: Drop non-feature columns ──────────────────────────────────────────
drop_cols = ['flow_id', 'source_ip', 'dest_ip', 'label']
drop_cols = [col for col in drop_cols if col in icmp_covert.columns]
icmp_covert.drop(columns=drop_cols, inplace=True)
print(f"Dropped columns: {drop_cols}")

# ── Step 5: Add remaining label columns ───────────────────────────────────────
icmp_covert['label_binary']   = 1        # Covert traffic
icmp_covert['label_protocol'] = 'ICMP'   # Protocol

# ── Step 6: Verify ────────────────────────────────────────────────────────────
print(f"\n✅ Final shape: {icmp_covert.shape}")
print("\n✅ Technique distribution:")
print(icmp_covert['label_technique'].value_counts().sort_index())
print("\n✅ Label summary:")
print(icmp_covert[['label_binary', 'label_protocol', 'label_technique']].value_counts())
print("\n✅ Sample rows:")
print(icmp_covert[['label_binary', 'label_protocol', 'label_technique']].head(10))

# ── Step 7: Save ──────────────────────────────────────────────────────────────
icmp_covert.to_csv("icmp_covert_labeled.csv", index=False)
print("\n✅ Saved as icmp_covert_labeled.csv")

# **UDP**

In [ ]:
udp_normal = pd.read_csv("/content/drive/MyDrive/Protocol-Wise/Protocol-Wise/UDP/Self-Generated/Normal/UDP_Normal.csv")

In [ ]:
udp_normal.head(3)

In [ ]:
print(f"Original shape: {udp_normal.shape}")
print(f"Columns: {udp_normal.columns.tolist()}")

# ── Step 1: Drop non-feature columns ─────────────────────────────────────────
drop_cols = ['src_ip', 'dst_ip']
drop_cols = [col for col in drop_cols if col in udp_normal.columns]
udp_normal.drop(columns=drop_cols, inplace=True)
print(f"Dropped columns: {drop_cols}")

# ── Step 2: Drop any existing label columns ───────────────────────────────────
label_cols = [col for col in udp_normal.columns if 'label' in col.lower()]
udp_normal.drop(columns=label_cols, inplace=True)
print(f"Dropped label columns: {label_cols}")

# ── Step 3: Use all 1732 rows ─────────────────────────────────────────────────
udp_normal = udp_normal.reset_index(drop=True)
print(f"Shape after dropping: {udp_normal.shape}")

# ── Step 4: Add 3 label columns ───────────────────────────────────────────────
udp_normal['label_binary']    = 0      # Normal traffic
udp_normal['label_protocol']  = 'UDP'  # Protocol
udp_normal['label_technique'] = 0      # No technique (normal)

# ── Step 5: Verify ────────────────────────────────────────────────────────────
print(f"\n✅ Final shape: {udp_normal.shape}")
print("\n✅ Label summary:")
print(udp_normal[['label_binary', 'label_protocol', 'label_technique']].value_counts())
print("\n✅ Sample rows:")
print(udp_normal[['label_binary', 'label_protocol', 'label_technique']].head(5))

# ── Step 6: Save ──────────────────────────────────────────────────────────────
udp_normal.to_csv("udp_normal_labeled.csv", index=False)
print("\n✅ Saved as udp_normal_labeled.csv")

In [ ]:
#UDP -Covert

udp_covert = pd.read_csv("/content/drive/MyDrive/Protocol-Wise/Protocol-Wise/UDP/Self-Generated/Covert/UDP_covert.csv")

In [ ]:
udp_covert.head(3)

In [ ]:
print(f"Original shape: {udp_covert.shape}")

# ── Step 1: Drop non-feature & existing label columns ────────────────────────
drop_cols = ['src_ip', 'dst_ip', 'label']
drop_cols = [col for col in drop_cols if col in udp_covert.columns]
udp_covert.drop(columns=drop_cols, inplace=True)
print(f"Dropped columns: {drop_cols}")
print(f"Shape after dropping: {udp_covert.shape}")

# ── Step 2: Define Feature Groups ────────────────────────────────────────────
header_features = [
    'fwd_pkt_len_max', 'fwd_pkt_len_min', 'fwd_pkt_len_mean', 'fwd_pkt_len_std',
    'pkt_len_max', 'pkt_len_min', 'pkt_len_mean', 'pkt_len_std',
    'fwd_header_len'
]

timing_features = [
    'flow_duration', 'flow_iat_mean', 'flow_iat_max', 'flow_iat_min', 'flow_iat_std',
    'fwd_iat_tot', 'fwd_iat_mean', 'fwd_iat_max', 'fwd_iat_min', 'fwd_iat_std'
]

blending_features = [
    'src_port', 'dst_port', 'tot_fwd_pkts', 'totlen_fwd_pkts',
    'flow_byts_s', 'flow_pkts_s', 'fwd_pkts_s',
    'subflow_fwd_pkts', 'subflow_fwd_byts'
]

# ── Step 3: Filter to only available columns (safety check) ──────────────────
header_features   = [f for f in header_features   if f in udp_covert.columns]
timing_features   = [f for f in timing_features   if f in udp_covert.columns]
blending_features = [f for f in blending_features if f in udp_covert.columns]

print(f"\nUsing features → Header: {len(header_features)} | "
      f"Timing: {len(timing_features)} | Blending: {len(blending_features)}")

# ── Step 4: Standardize all features ─────────────────────────────────────────
scaler = StandardScaler()
all_features = list(dict.fromkeys(header_features + timing_features + blending_features))
udp_scaled = pd.DataFrame(
    scaler.fit_transform(udp_covert[all_features]),
    columns=all_features
)

# ── Step 5: Compute Anomaly Score per Group ───────────────────────────────────
udp_covert['score_header']   = udp_scaled[header_features].abs().mean(axis=1)
udp_covert['score_timing']   = udp_scaled[timing_features].abs().mean(axis=1)
udp_covert['score_blending'] = udp_scaled[blending_features].abs().mean(axis=1)

# ── Step 6: Assign technique based on highest anomaly score ──────────────────
# Technique 1 = Header Manipulation
# Technique 2 = Time Obfuscation
# Technique 3 = Flow Blending
def assign_technique(row):
    scores = {
        1: row['score_header'],
        2: row['score_timing'],
        3: row['score_blending']
    }
    return max(scores, key=scores.get)

udp_covert['label_technique'] = udp_covert.apply(assign_technique, axis=1)

# ── Step 7: Add remaining label columns ──────────────────────────────────────
udp_covert['label_binary']   = 1      # Covert traffic
udp_covert['label_protocol'] = 'UDP'  # Protocol

# ── Step 8: Drop scoring helper columns ──────────────────────────────────────
udp_covert.drop(columns=['score_header', 'score_timing', 'score_blending'], inplace=True)

# ── Step 9: Verify ────────────────────────────────────────────────────────────
print(f"\n✅ Final shape: {udp_covert.shape}")
print("\n✅ Technique distribution:")
print(udp_covert['label_technique'].value_counts().sort_index())
print("\n✅ Label summary:")
print(udp_covert[['label_binary', 'label_protocol', 'label_technique']].value_counts())
print("\n✅ Sample rows:")
print(udp_covert[['label_binary', 'label_protocol', 'label_technique']].head(10))

# ── Step 10: Save ─────────────────────────────────────────────────────────────
udp_covert.to_csv("udp_covert_labeled.csv", index=False)
print("\n✅ Saved as udp_covert_labeled.csv")

In [ ]:
#Handling Class imbalnce For Header Manipulation (Labael Technique 1)

# ============================================================
# SMOTE Fix for UDP Covert Dataset — Label 1 Class Imbalance
# FYP: Transformer-Based Network Steganography Detector
# Run this in Google Colab
# ============================================================

# ── STEP 1: Install dependencies ─────────────────────────────
!pip install imbalanced-learn -q

# ── STEP 2: Imports ──────────────────────────────────────────
import pandas as pd
import numpy as np
from imblearn.over_sampling import SMOTE
from collections import Counter
import matplotlib.pyplot as plt
from google.colab import files

# ── STEP 3: load your file ─────────────────────────────────
df = pd.read_csv("/content/udp_covert_labeled.csv")

print(f"\n✅ Loaded: {df.shape[0]} rows, {df.shape[1]} columns")
print("\n📊 Before SMOTE — label_technique distribution:")
print(df['label_technique'].value_counts().sort_index())

# ── STEP 4: Separate features and label ──────────────────────
# Drop non-numeric / label columns for SMOTE
drop_cols = ['label_technique', 'label_binary', 'label_protocol']
X = df.drop(columns=drop_cols)
y = df['label_technique']

# Keep meta columns to re-attach later
meta = df[['label_binary', 'label_protocol']].copy()

# ── STEP 5: Apply SMOTE ───────────────────────────────────────
# Target: balance all classes to match the majority class (Label 2 = 795)
majority_count = y.value_counts().max()

smote = SMOTE(
    sampling_strategy={
        1: majority_count,   # Oversample Label 1 (Header Manipulation) → 795
        3: majority_count    # Oversample Label 3 (Flow Blending) → 795 (optional, already 414)
    },
    random_state=42,
    k_neighbors=5           # Safe default; reduce to 3 if Label 1 < 10 samples
)

X_resampled, y_resampled = smote.fit_resample(X, y)

print(f"\n✅ After SMOTE — label_technique distribution:")
print(pd.Series(y_resampled).value_counts().sort_index())
print(f"\nTotal rows after SMOTE: {len(X_resampled)}")

# ── STEP 6: Rebuild DataFrame ────────────────────────────────
df_resampled = pd.DataFrame(X_resampled, columns=X.columns)
df_resampled['label_technique'] = y_resampled.values
df_resampled['label_binary'] = 1          # All covert
df_resampled['label_protocol'] = 'UDP'

print("\n📋 Resampled DataFrame shape:", df_resampled.shape)
print(df_resampled['label_technique'].value_counts().sort_index())

# ── STEP 7: Validate — check no NaN or Inf ───────────────────
nan_count = df_resampled.isnull().sum().sum()
inf_count = np.isinf(df_resampled.select_dtypes(include=np.number)).sum().sum()
print(f"\n🔍 NaN values: {nan_count}")
print(f"🔍 Inf values: {inf_count}")

if nan_count > 0 or inf_count > 0:
    df_resampled.replace([np.inf, -np.inf], np.nan, inplace=True)
    df_resampled.fillna(df_resampled.median(numeric_only=True), inplace=True)
    print("⚠️  NaN/Inf found and fixed with median imputation.")
else:
    print("✅ No NaN or Inf — data is clean.")

# ── STEP 8: Plot before vs after ─────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

before_counts = df['label_technique'].value_counts().sort_index()
after_counts  = pd.Series(y_resampled).value_counts().sort_index()

label_names = {1: 'Header\nManip', 2: 'Timing\nObfusc', 3: 'Flow\nBlend'}
x_labels = [label_names[i] for i in before_counts.index]

axes[0].bar(x_labels, before_counts.values, color=['#e74c3c','#3498db','#2ecc71'])
axes[0].set_title('Before SMOTE', fontsize=13, fontweight='bold')
axes[0].set_ylabel('Sample Count')
for i, v in enumerate(before_counts.values):
    axes[0].text(i, v + 5, str(v), ha='center', fontweight='bold')

x_labels_after = [label_names[i] for i in after_counts.index]
axes[1].bar(x_labels_after, after_counts.values, color=['#e74c3c','#3498db','#2ecc71'])
axes[1].set_title('After SMOTE', fontsize=13, fontweight='bold')
axes[1].set_ylabel('Sample Count')
for i, v in enumerate(after_counts.values):
    axes[1].text(i, v + 5, str(v), ha='center', fontweight='bold')

plt.suptitle('UDP Covert Dataset — SMOTE Class Balance Fix', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('smote_balance_plot.png', dpi=150, bbox_inches='tight')
plt.show()
print("📊 Plot saved as smote_balance_plot.png")

# ── STEP 9: Save balanced dataset ────────────────────────────
output_name = 'udp_covert_labeled_smote.csv'
df_resampled.to_csv(output_name, index=False)
print(f"\n💾 Saved: {output_name}")

# ── STEP 10: Download ─────────────────────────────────────────
files.download(output_name)
files.download('smote_balance_plot.png')
print("\n✅ Done! Both files downloaded.")

# ── STEP 11: Final Summary ────────────────────────────────────
print("\n" + "="*50)
print("FINAL SUMMARY")
print("="*50)
print(f"Original rows   : {len(df)}")
print(f"Resampled rows  : {len(df_resampled)}")
print(f"Label 1 (Header): {(df_resampled['label_technique']==1).sum()} rows")
print(f"Label 2 (Timing): {(df_resampled['label_technique']==2).sum()} rows")
print(f"Label 3 (Flow)  : {(df_resampled['label_technique']==3).sum()} rows")
print("="*50)

# **DNS-Tunneling**

In [ ]:
dns_normal = pd.read_csv("/content/drive/MyDrive/Protocol-Wise/Protocol-Wise/UDP/DNS-Tunneling/Benign_DNS.csv")

In [ ]:
dns_normal.head(3)

In [ ]:
print(f"Original shape: {dns_normal.shape}")

# ── Step 1: Drop non-feature columns ─────────────────────────────────────────
drop_cols = ['flow_id', 'timestamp', 'src_ip', 'dst_ip', 'label']
drop_cols = [col for col in drop_cols if col in dns_normal.columns]
dns_normal.drop(columns=drop_cols, inplace=True)
print(f"Dropped columns: {drop_cols}")

# ── Step 2: Drop any other existing label columns ─────────────────────────────
label_cols = [col for col in dns_normal.columns if 'label' in col.lower()]
dns_normal.drop(columns=label_cols, inplace=True)
print(f"Dropped additional label columns: {label_cols}")

print(f"Shape after dropping: {dns_normal.shape}")

# ── Step 3: Balance to first 7,000 rows ──────────────────────────────────────
dns_normal = dns_normal.head(7000).reset_index(drop=True)
print(f"After balancing: {len(dns_normal)} rows")

# ── Step 4: Add 3 label columns ───────────────────────────────────────────────
dns_normal['label_binary']    = 0      # Normal traffic
dns_normal['label_protocol']  = 'DNS'  # Protocol
dns_normal['label_technique'] = 0      # No technique (normal)

# ── Step 5: Verify ────────────────────────────────────────────────────────────
print(f"\n✅ Final shape: {dns_normal.shape}")
print("\n✅ Label summary:")
print(dns_normal[['label_binary', 'label_protocol', 'label_technique']].value_counts())
print("\n✅ Sample rows:")
print(dns_normal[['label_binary', 'label_protocol', 'label_technique']].head(5))

# ── Step 6: Save ──────────────────────────────────────────────────────────────
dns_normal.to_csv("dns_normal_labeled.csv", index=False)
print("\n✅ Saved as dns_normal_labeled.csv")

In [ ]:
#DNS - Covert

dns_covert = pd.read_csv("/content/DNS_Tunneling_Covert_traffic.csv", on_bad_lines='skip', engine='python')

In [ ]:
dns_covert.head(3)

In [ ]:

df = pd.read_csv('/content/DNS_Tunneling_Covert_traffic.csv', on_bad_lines='skip', engine='python')

# ── Step 1: Drop non-feature and existing label columns ────────────────────────
drop_cols = ['flow_id', 'timestamp', 'src_ip', 'dst_ip', 'label']
drop_cols = [col for col in drop_cols if col in df.columns]
df.drop(columns=drop_cols, inplace=True)

# ── Step 2: Add label_binary and label_protocol ──────────────────────────────
df['label_binary']   = 1      # Covert traffic
df['label_protocol'] = 'DNS'  # Protocol

# ── Step 3: Arbitrarily assign label_technique for balancing (placeholder) ──
# This step is added because the raw DNS covert data does not have specific
# technique labels (1, 2, 3). For demonstration, we divide the data
# into three parts and assign technique labels arbitrarily to enable
# the sampling across 'techniques'. In a real scenario, these would be
# derived from feature analysis or more granular labels.
total_rows = len(df)
chunk_size = total_rows // 3
df.loc[:chunk_size-1, 'label_technique'] = 1
df.loc[chunk_size:2*chunk_size-1, 'label_technique'] = 2
df.loc[2*chunk_size:, 'label_technique'] = 3
df['label_technique'] = df['label_technique'].astype(int)

print("Original shape:", df.shape)
print(df['label_technique'].value_counts())

# # ── Option 1: Random sample (if already balanced) ──────────
# df_sampled = df.sample(n=10000, random_state=42)

# # ── Option 2: Stratified sample (keeps technique ratio) ────
# df_sampled = df.groupby('label_technique', group_keys=False).apply(
#     lambda x: x.sample(frac=10000/len(df), random_state=42)
# ).reset_index(drop=True)

# ── Option 3: Exactly balanced — 3333 per technique ────────
df_sampled = df.groupby('label_technique', group_keys=False).apply(
    lambda x: x.sample(n=3333, random_state=42)
).reset_index(drop=True)

print("Sampled shape:", df_sampled.shape)
print(df_sampled['label_technique'].value_counts())

df_sampled.to_csv('dns_covert_10k.csv', index=False)

In [ ]:
import pandas as pd

df = pd.read_csv('/content/dns_covert_10k.csv')
print("Original shape:", df.shape)
print("Before:\n", df['label_technique'].value_counts())

# ── Exactly balanced — 667 per technique (667 x 3 = 2001 ≈ 2000) ──
df_sampled = df.groupby('label_technique', group_keys=False).apply(
    lambda x: x.sample(n=667, random_state=42)
).reset_index(drop=True)

print("\nAfter:\n", df_sampled['label_technique'].value_counts())
print("Total rows:", len(df_sampled))

df_sampled.to_csv('dns_covert_2k.csv', index=False)
print("Saved: dns_covert_2k.csv")

# **All Labeled files combined (Final Master Raw Data)**

In [ ]:
import numpy as np
import re
import ast
from collections import Counter
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
import torch

In [ ]:
# Load files
tcp_normal = pd.read_csv("/content/drive/MyDrive/3_levels_Protocol_Wise/TCP/tcp_normal_labeled.csv")
tcp_covert = pd.read_csv("/content/drive/MyDrive/3_levels_Protocol_Wise/TCP/tcp_stego_labeled.csv")

icmp_normal = pd.read_csv("/content/drive/MyDrive/3_levels_Protocol_Wise/ICMP/icmp_normal_labeled.csv")
icmp_covert = pd.read_csv("/content/drive/MyDrive/3_levels_Protocol_Wise/ICMP/icmp_covert_labeled.csv")

dns_normal = pd.read_csv("/content/drive/MyDrive/3_levels_Protocol_Wise/DNS/dns_normal_labeled.csv")
dns_covert = pd.read_csv("/content/drive/MyDrive/3_levels_Protocol_Wise/DNS/dns_covert_labeled.csv")

udp_normal = pd.read_csv("/content/drive/MyDrive/3_levels_Protocol_Wise/UDP/udp_normal_labeled.csv")
udp_covert = pd.read_csv("/content/drive/MyDrive/3_levels_Protocol_Wise/UDP/udp_covert_labeled.csv")

In [ ]:
tcp_normal["dataset_name"] = "tcp_normal"
tcp_covert["dataset_name"] = "tcp_covert"

icmp_normal["dataset_name"] = "icmp_normal"
icmp_covert["dataset_name"] = "icmp_covert"

dns_normal["dataset_name"] = "dns_normal"
dns_covert["dataset_name"] = "dns_covert"

udp_normal["dataset_name"] = "udp_normal"
udp_covert["dataset_name"] = "udp_covert"

In [ ]:
final_df = pd.concat(
    [
        tcp_normal, tcp_covert,
        icmp_normal, icmp_covert,
        dns_normal, dns_covert,
        udp_normal, udp_covert
    ],
    axis=0,
    ignore_index=True,
    sort=False   # VERY IMPORTANT
)

In [ ]:
print("Final shape:", final_df.shape)
print("Total columns:", len(final_df.columns))
print(final_df.columns)

In [ ]:
final_df.to_csv(
    "/content/drive/MyDrive/FINAL_MASTER_RAW_DATASET.csv",
    index=False
)

print("Raw combined dataset saved successfully.")

In [ ]:
print(final_df.isnull().sum().sum())
print(final_df.head(5))
print(final_df.tail(5))

In [ ]:
print(final_df["label_protocol"].value_counts())

In [ ]:
final_df[final_df["label_protocol"] == "TCP"].head()

# **PreProcessing**

In [ ]:
df = pd.read_csv("/content/drive/MyDrive/Network_steganography_Project/FINAL_MASTER_RAW_DATASET.csv")

In [ ]:
# =====================================================
# COMPLETE PREPROCESSING PIPELINE
# Transformer-Based Network Steganography Detector
# =====================================================

import pandas as pd
import numpy as np
import math
from collections import Counter
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.utils import resample

# =====================================================
# 1. LOAD DATASET
# =====================================================

df = pd.read_csv("/content/drive/MyDrive/Network_steganography_Project/FINAL_MASTER_RAW_DATASET.csv", low_memory=False)

print("Original Shape:", df.shape)

# =====================================================
# 2. REMOVE UNNECESSARY COLUMNS
# =====================================================

cols_to_drop = ["dataset_name"]

for col in cols_to_drop:
    if col in df.columns:
        df.drop(columns=col, inplace=True)

print("Removed dataset_name column (if existed)")

# =====================================================
# 3. HANDLE DUPLICATE DOMAIN LENGTH FEATURE
# =====================================================

if "dns_domain_name_length" in df.columns and "dns_domain_name_len" in df.columns:

    print("Both domain length columns exist. Checking if identical...")

    if (df["dns_domain_name_length"] == df["dns_domain_name_len"]).all():
        print("Columns are identical. Removing dns_domain_name_length")
        df.drop(columns=["dns_domain_name_length"], inplace=True)
    else:
        print("Columns are different. Keeping both.")

# =====================================================
# 4. DNS LEXICAL FEATURE ENGINEERING
# =====================================================

def shannon_entropy(text):

    if len(text) == 0:
        return 0

    freq = Counter(text)
    probs = [v / len(text) for v in freq.values()]
    return -sum(p * math.log2(p) for p in probs)


def extract_domain_features(domain):

    domain = str(domain)

    length = len(domain)
    digits = sum(c.isdigit() for c in domain)
    letters = sum(c.isalpha() for c in domain)
    special = sum(not c.isalnum() for c in domain)

    unique_chars = len(set(domain))
    entropy = shannon_entropy(domain)

    return pd.Series([
        length,
        digits,
        letters,
        special,
        unique_chars,
        entropy,
        digits/length if length > 0 else 0,
        special/length if length > 0 else 0
    ])


dns_domain_cols = [
    "dns_domain_name",
    "uni_gram_domain_name",
    "bi_gram_domain_name",
    "tri_gram_domain_name"
]


for col in dns_domain_cols:

    if col in df.columns:

        df[col] = df[col].fillna("").astype(str)

        df[[f"{col}_len",
            f"{col}_digit_count",
            f"{col}_letter_count",
            f"{col}_special_count",
            f"{col}_unique_chars",
            f"{col}_entropy",
            f"{col}_digit_ratio",
            f"{col}_special_ratio"]] = df[col].apply(extract_domain_features)


# Drop heavy text columns
df.drop(columns=[c for c in dns_domain_cols if c in df.columns], inplace=True)

print("After DNS Feature Engineering:", df.shape)

# =====================================================
# 5. SIMPLE ENCODING FOR LOW CARDINALITY DNS FIELDS
# =====================================================

categorical_cols = [
    "dns_top_level_domain",
    "dns_second_level_domain",
    "ans_resource_record_type",
    "ans_resource_record_class"
]

for col in categorical_cols:

    if col in df.columns:

        df[col] = df[col].fillna("unknown").astype(str)

        le = LabelEncoder()
        df[col] = le.fit_transform(df[col])

print("Categorical DNS fields encoded")

# =====================================================
# 6. ENCODE PROTOCOL LABEL
# =====================================================

protocol_encoder = LabelEncoder()

df["label_protocol_enc"] = protocol_encoder.fit_transform(df["label_protocol"])

df.drop(columns=["label_protocol"], inplace=True)

print("Protocol encoded")

# =====================================================
# 7. BALANCE DATA PER PROTOCOL + BINARY
# =====================================================

balanced_list = []

for protocol in df["label_protocol_enc"].unique():

    subset = df[df["label_protocol_enc"] == protocol]

    normal = subset[subset["label_binary"] == 0]
    covert = subset[subset["label_binary"] == 1]

    if len(normal) < len(covert):

        normal = resample(
            normal,
            replace=True,
            n_samples=len(covert),
            random_state=42
        )

    else:

        normal = normal.sample(len(covert), random_state=42)

    balanced_protocol = pd.concat([normal, covert])
    balanced_list.append(balanced_protocol)


df_balanced = pd.concat(balanced_list)

print("After Protocol + Binary Balancing:")
print(df_balanced["label_binary"].value_counts())

# =====================================================
# 8. BALANCE PER TECHNIQUE
# =====================================================

min_tech = df_balanced["label_technique"].value_counts().min()

tech_list = []

for tech, group in df_balanced.groupby("label_technique"):

    tech_list.append(group.sample(min_tech, random_state=42))


df_balanced = pd.concat(tech_list)

print("After Technique Balance:")
print(df_balanced["label_technique"].value_counts())

# =====================================================
# 9. CLEAN DATA AND FORCE NUMERIC
# =====================================================

df_balanced.replace([np.inf, -np.inf], np.nan, inplace=True)

df_balanced.fillna(0, inplace=True)

label_cols = [
    "label_binary",
    "label_protocol_enc",
    "label_technique"
]

for col in df_balanced.columns:

    if col not in label_cols:

        df_balanced[col] = pd.to_numeric(df_balanced[col], errors='coerce')


df_balanced.fillna(0, inplace=True)

print("Data cleaned")

# =====================================================
# 10. FEATURE SCALING
# =====================================================

feature_cols = [c for c in df_balanced.columns if c not in label_cols]

scaler = StandardScaler()

df_balanced[feature_cols] = scaler.fit_transform(df_balanced[feature_cols])

print("Feature scaling completed")

# =====================================================
# 11. SAVE FINAL PREPROCESSED DATASET
# =====================================================

df_balanced.to_csv("preprocessed_master_dataset.csv", index=False)

print("Saved: preprocessed_master_dataset.csv")
print("Final Shape:", df_balanced.shape)

# =====================================================
# 12. TRAIN TEST SPLIT (STRATIFIED BY TECHNIQUE)
# =====================================================

X = df_balanced[feature_cols]

y_binary = df_balanced["label_binary"]
y_protocol = df_balanced["label_protocol_enc"]
y_technique = df_balanced["label_technique"]


X_train, X_test, yb_train, yb_test, yp_train, yp_test, yt_train, yt_test = train_test_split(
    X,
    y_binary,
    y_protocol,
    y_technique,
    test_size=0.2,
    random_state=42,
    stratify=y_technique
)


train_df = X_train.copy()
train_df["label_binary"] = yb_train.values
train_df["label_protocol_enc"] = yp_train.values
train_df["label_technique"] = yt_train.values


test_df = X_test.copy()
test_df["label_binary"] = yb_test.values
test_df["label_protocol_enc"] = yp_test.values
test_df["label_technique"] = yt_test.values


train_df.to_csv("train_dataset.csv", index=False)
test_df.to_csv("test_dataset.csv", index=False)

print("Train Shape:", train_df.shape)
print("Test Shape:", test_df.shape)

print("Preprocessing Completed Successfully 🚀")

In [ ]:
import os

print(os.listdir())

In [ ]:
import pandas as pd

# Load dataset
new= pd.read_csv("/content/preprocessed_master_dataset.csv")

# 1️⃣ Shape (rows, columns)
print("Dataset Shape (rows, columns):", new.shape)

# 2️⃣ Number of rows
print("Total Rows:", new.shape[0])

# 3️⃣ Total Columns
print("Total Columns:", new.shape[1])

# 4️⃣ Feature Columns Only (excluding labels)
label_cols = ["label_binary", "label_protocol_enc", "label_technique"]
feature_cols = [col for col in new.columns if col not in label_cols]

print("Total Feature Columns:", len(feature_cols))

In [ ]:
new.columns.to_list()

In [ ]:
print(new["character_distribution"].dtype)

In [ ]:
non_numeric = new.select_dtypes(exclude=['number'])
print(non_numeric.columns)

In [ ]:
print("Final Shape:", new.shape)

# Check remaining non-numeric columns
non_numeric = new.select_dtypes(exclude=['number'])
print("Non-numeric columns remaining:")
print(non_numeric.columns)

In [ ]:
new.head()

In [ ]:
print(new["label_binary"].value_counts())

In [ ]:
normal = (new["label_binary"] == 0).sum()
covert = (new["label_binary"] == 1).sum()

print("Total Normal Samples :", normal)
print("Total Covert Samples :", covert)
print("Total Dataset Samples:", len(new))

In [ ]:
print(new["label_binary"].value_counts(normalize=True) * 100)

In [ ]:
import os

base_path = "/content/drive/MyDrive/Network_steganography_Project"
os.makedirs(base_path, exist_ok=True)

In [ ]:
new.to_csv(f"{base_path}/preprocessed_master_dataset.csv", index=False)
print("Saved full dataset")

In [ ]:
train_df.to_csv(f"{base_path}/train_dataset.csv", index=False)
test_df.to_csv(f"{base_path}/test_dataset.csv", index=False)

In [ ]:
print(os.listdir(base_path))

# **Visualizations**

In [ ]:

import seaborn as sns
import matplotlib.pyplot as plt

plt.figure(figsize=(6,4))
sns.countplot(x='label_binary', data=new)
plt.title("Binary Class Distribution")
plt.show()

In [ ]:
plt.figure(figsize=(6,4))
sns.countplot(x='label_technique', data=new)
plt.title("Technique Class Distribution")
plt.show()

In [ ]:
plt.figure(figsize=(6,4))
sns.countplot(x='label_protocol_enc', data=new)
plt.title("Protocol Class Distribution")
plt.show()

In [ ]:
from sklearn.ensemble import RandomForestClassifier

X = new.drop(columns=['label_binary','label_technique','label_protocol_enc'])
y = new['label_binary']

rf = RandomForestClassifier(n_estimators=100)
rf.fit(X, y)

importances = pd.Series(rf.feature_importances_, index=X.columns)
top_features = importances.sort_values(ascending=False).head(20)

plt.figure(figsize=(8,6))
top_features.plot(kind='barh')
plt.title("Top 20 Important Features (Binary)")
plt.show()